# Topic Modeling with BERTopic — Parameter Search & Evaluation (iGEM Teams)

Loads the pre-computed **iGEM Teams** embeddings, runs a grid search over
key UMAP/HDBSCAN parameters, evaluates each configuration with **C_v
coherence**, **topic diversity**, and **DBCV**, selects the best model,
optionally reassigns outliers, and saves the results to today's run folder,
`assets/<date>/02/`.

> **Recommended** — this is the notebook used in the associated publication.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 02-topic_model/, where the aux/ package
# and setup_run.py reside; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import set_seed
from setup_run import setup
from aux.topic_modeling import load_corpus, save_topic_outputs
from aux.evaluation import grid_search

set_seed()

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
RUN             = setup(corpus="teams")  # today's run folder: assets/<date>/02/
EMBEDDINGS_FILE = "teams_embeddings.npy"
CORPUS_FILE     = "teams_corpus.txt"
ID_COL          = "UT"
PREFIX          = "teams"

PARAM_GRID = {
    "min_cluster_size": [8, 10, 15, 20],
    "umap_n_neighbors": [10, 15, 25],
    "umap_n_components": [5, 10],
}

# Reassign all iGEM noise documents to their nearest topic (see section 3).
REDUCE_OUTLIERS = True

## 1. Load embeddings and corpus

In [3]:
embeddings, corpus = load_corpus(RUN, EMBEDDINGS_FILE, CORPUS_FILE)
docs = corpus["text"].tolist()
print(f"Teams: {embeddings.shape[0]:,} docs, {embeddings.shape[1]} dims")

Teams: 3,811 docs, 384 dims


## 2. Grid search

Fit and evaluate a BERTopic model for every parameter combination.

In [4]:
results, best = grid_search(docs, embeddings, PARAM_GRID, label="Teams")
results

[Teams] 1/24  mcs=8, nn=10, nc=5 ... topics=169, C_v=0.5372, div=0.7142, DBCV=0.1628
[Teams] 2/24  mcs=8, nn=10, nc=10 ... topics=168, C_v=0.5348, div=0.7202, DBCV=0.2030
[Teams] 3/24  mcs=8, nn=15, nc=5 ... topics=150, C_v=0.5565, div=0.7073, DBCV=0.1836
[Teams] 4/24  mcs=8, nn=15, nc=10 ... topics=154, C_v=0.5589, div=0.7065, DBCV=0.1709
[Teams] 5/24  mcs=8, nn=25, nc=5 ... topics=145, C_v=0.5517, div=0.6986, DBCV=0.1926
[Teams] 6/24  mcs=8, nn=25, nc=10 ... topics=145, C_v=0.5488, div=0.7048, DBCV=0.1553
[Teams] 7/24  mcs=10, nn=10, nc=5 ... topics=144, C_v=0.5406, div=0.6785, DBCV=0.1705
[Teams] 8/24  mcs=10, nn=10, nc=10 ... topics=132, C_v=0.5406, div=0.6424, DBCV=0.2117
[Teams] 9/24  mcs=10, nn=15, nc=5 ... topics=117, C_v=0.5350, div=0.6145, DBCV=0.1724
[Teams] 10/24  mcs=10, nn=15, nc=10 ... topics=125, C_v=0.5343, div=0.6448, DBCV=0.1641
[Teams] 11/24  mcs=10, nn=25, nc=5 ... topics=111, C_v=0.5448, div=0.5982, DBCV=0.1609
[Teams] 12/24  mcs=10, nn=25, nc=10 ... topics=116, C

,min_cluster_size,n_neighbors,n_components,n_topics,outlier_frac,coherence_cv,diversity,dbcv
0,8,15,10,154,0.2288,0.5589,0.7065,0.1709
1,8,15,5,150,0.2438,0.5565,0.7073,0.1836
2,8,25,5,145,0.2642,0.5517,0.6986,0.1926
3,8,25,10,145,0.2821,0.5488,0.7048,0.1553
4,10,25,5,111,0.2711,0.5448,0.5982,0.1609
5,10,25,10,116,0.2965,0.5414,0.6241,0.1433
6,10,10,5,144,0.2018,0.5406,0.6785,0.1705
7,10,10,10,132,0.2057,0.5406,0.6424,0.2117
8,8,10,5,169,0.1881,0.5372,0.7142,0.1628
9,10,15,5,117,0.2301,0.5350,0.6145,0.1724


In [5]:
print("Best configuration:")
for k in ["min_cluster_size", "n_neighbors", "n_components", "n_topics",
          "coherence_cv", "diversity", "dbcv", "outlier_frac"]:
    print(f"  {k:16s} = {best[k]}")

Best configuration:
  min_cluster_size = 8
  n_neighbors      = 15
  n_components     = 10
  n_topics         = 154
  coherence_cv     = 0.5589
  diversity        = 0.7065
  dbcv             = 0.1709
  outlier_frac     = 0.2288


## 3. Reduce outliers (optional)

HDBSCAN labels documents that fall outside any dense cluster as topic **−1**
(noise). While that is acceptable for the SynBio literature (some papers may be
genuinely off-topic), every iGEM team project is by definition related to
synthetic biology — its text may simply be too short or idiosyncratic to land in
a cluster. BERTopic's `reduce_outliers` (strategy `"embeddings"`, threshold `0`)
reassigns **all** noise documents to their nearest topic by cosine similarity,
without retraining the model.

Controlled by `REDUCE_OUTLIERS` in the config above (enabled for teams,
disabled for papers).

In [6]:
model = best["model"]
topics = list(best["topics"])

if REDUCE_OUTLIERS:
    before = sum(1 for t in topics if t == -1)
    topics = model.reduce_outliers(
        docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0,
    )
    model.update_topics(docs, topics=topics)
    after = sum(1 for t in topics if t == -1)
    print(f"Outliers: {before:,} → {after:,}")
else:
    print("Outlier reduction disabled — keeping HDBSCAN noise labels.")

2026-08-03 15:19:52,973 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers: 872 → 0


## 4. Save best model, outputs, and grid-search results

In [7]:
save_topic_outputs(RUN, model, corpus, topics, ID_COL, PREFIX)
results.to_csv(RUN.out(f"{PREFIX}_grid_search.txt"), sep="\t", index=False)

print(f"Saved → {RUN.dir}")
for f in sorted(RUN.dir.glob(f"{PREFIX}_*")):
    print(f"  {f.name}")

2026-08-03 15:19:55,298 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/topic_models
  teams_doc_topics.txt
  teams_grid_search.txt
  teams_topic_info.txt
  teams_topic_model
  teams_topic_names.txt
